# Notebook : extraction des références composants

**Projet :** Études Pratiques — Lecture automatique de schémas électroniques  
**Équipe :** Maïwenn Lepéroux · Ilyas El Maaroufi · Rayane El Bouhali · Victoire Peltier · Louis Richard (2025-2026)

## ▶  Comment lancer ce notebook

**Toujours lancer Jupyter depuis la racine du projet**, jamais depuis ce sous-dossier :

```bash
cd ~/GitHub/etudes-pratiques
jupyter notebook parts/Lecture_des_composants/notebook/NotebookLC.ipynb
```

Ou avec JupyterLab :
```bash
cd ~/GitHub/etudes-pratiques
jupyter lab parts/Lecture_des_composants/notebook/NotebookLC.ipynb
```

> **Pourquoi depuis la racine ?**  
> Tous les chemins de modèles (`models/extraction/best.pt`, `parts/Lecture_table/dev/bestr.pt`) et d'images de test sont relatifs à `etudes-pratiques/`. Si vous lancez depuis ce dossier, tous les `open()` et `YOLO(path)` trouveront leurs fichiers sans modification.

In [1]:
import os
from pathlib import Path

# Remonte automatiquement jusqu'à la racine du projet
# (le dossier qui contient models/extraction/best.pt ET parts/)
# Fonctionne quel que soit le répertoire depuis lequel Jupyter est lancé.
_root = Path.cwd()
for _ in range(6):
    if (_root / 'models' / 'extraction' / 'best.pt').exists() and (_root / 'parts').exists():
        break
    _root = _root.parent
else:
    # Fallback : chemin absolu si la remontée automatique échoue
    _root = Path.home() / 'GitHub' / 'etudes-pratiques'

os.chdir(_root)
print(f'Répertoire de travail : {Path.cwd()}')

if not (_root / 'models' / 'extraction' / 'best.pt').exists():
    print('Modèle introuvable — vérifiez votre arborescence.')
else:
    print('Tous les chemins relatifs fonctionnent depuis ici.')


Répertoire de travail : /home/rayane/GitHub/etudes-pratiques
Tous les chemins relatifs fonctionnent depuis ici.


### Permettre à l'utilisateur d'uploader directement son document

Le widget ci-dessous affiche un bouton de téléversement. Le fichier PDF uploadé est automatiquement sauvegardé dans `notebook/uploaded_images/` et son chemin est mémorisé dans `pdf_path` pour être utilisé par la suite.

In [2]:
import ipywidgets as widgets
from IPython.display import display
import os

upload_dir = 'notebook/uploaded_images'
os.makedirs(upload_dir, exist_ok=True)

uploader = widgets.FileUpload(accept='.pdf', multiple=False, description='Uploader PDF')
output   = widgets.Output()
pdf_path = set()

def on_upload(change):
    output.clear_output()
    with output:
        for name, data in uploader.value.items():
            save_path = os.path.join(upload_dir, name)
            with open(save_path, 'wb') as f:
                f.write(data['content'])
            pdf_path.add(save_path)
            print(f'✔  Fichier sauvegardé : {save_path}')

uploader.observe(on_upload, names='value')
display(uploader, output)

FileUpload(value=(), accept='.pdf', description='Uploader PDF')

Output()

## Transformer le notebook en application web

Ligne de commande (dans le répertoire racine du projet, avec Voilà installé) :

```bash
voila parts/Lecture_des_composants/notebook/NotebookLC.ipynb
```

Puis, dans un navigateur : `http://localhost:8866`

# Code assemblé : l'application

Cette partie analyse les fichiers au format PDF pour en extraire automatiquement trois types de zones :

- **Table** : tableau BOM (Bill of Materials), c'est-à-dire la liste des composants
- **Diagram** : schéma électronique
- **Board** : carte PCB (circuit imprimé)

Cette partie est composée de quatre fichiers qui s'articulent ensemble :

| Fichier | Rôle |
|---|---|
| `prediction.py` | Script autonome : détecte et sauvegarde toutes les zones d'un PDF |
| `merge.py` | Orchestrateur : sélectionne la meilleure zone par classe et lance l'interface |
| `test1.py` | Interface visuelle interactive : affiche et relie les composants détectés |
| `extract_easyocr.py` | OCR avancé v7 : YOLO + EasyOCR + merge de boîtes + correcteur OCR |

**Pipeline global :**

```
PDF → conversion en images (pdf2image) → détection de zones (YOLO best.pt)
    → sélection meilleure zone par classe → rotations
    → OCR des étiquettes (YOLO bestr.pt + Tesseract  OU  YOLO + EasyOCR v7)
    → interface interactive Matplotlib (clic + zoom)
```

Tous les fichiers source sont dans `parts/Lecture_table/dev/`.  
Les modèles sont dans `models/extraction/` et `parts/Lecture_table/dev/`.

---
## 1. `prediction.py` — Extraction brute des zones

Le script parcourt toutes les pages d'un PDF, détecte les zones via YOLO et sauvegarde chaque découpe en image PNG dans un dossier de sortie. Il ne filtre pas, ne choisit pas : il extrait tout.

**Usage standalone :**
```bash
python3 parts/Lecture_table/dev/prediction.py document.pdf /dossier/sortie/
```

### 1.1 Imports et chargement du modèle

Bibliothèques nécessaires : `pdf2image`, `numpy`, `cv2`, `ultralytics`.

On charge le modèle YOLO `models/extraction/best.pt`, entraîné spécifiquement pour reconnaître les trois classes dans des fiches techniques : `Table`, `Diagram`, `Board`.

Le dossier `IMAGES/` est créé s'il n'existe pas, pour recevoir les crops découpés.

In [3]:
pwd

'/home/rayane/GitHub/etudes-pratiques'

In [4]:
import os
import sys
import cv2
import numpy as np
from pathlib import Path
from pdf2image import convert_from_path
from ultralytics import YOLO

# Modèle YOLO pour la classification des zones (Table / Diagram / Board)
MODEL_EXTRACTION_PATH = 'models/extraction/best.pt'
model_extraction = YOLO(MODEL_EXTRACTION_PATH)
print(f'✔  Modèle chargé : {MODEL_EXTRACTION_PATH}')

# Dossier de sortie pour les crops
dir_path = 'parts/Lecture_des_composants/notebook/IMAGES'
os.makedirs(dir_path, exist_ok=True)

# Chemin du PDF — priorité à l'upload, sinon chemin par défaut
if pdf_path:
    save_path = next(iter(pdf_path))
else:
    save_path = 'notebook/uploaded_images/Accuphase-E202amp.pdf'

if not os.path.exists(save_path):
    print(f'⚠️  Fichier PDF introuvable : {save_path}')
else:
    print(f'✔  PDF source : {save_path}')

✔  Modèle chargé : models/extraction/best.pt
✔  PDF source : notebook/uploaded_images/Accuphase-E202amp.pdf


### 1.2 Conversion du PDF en images

`pdf2image` transforme chaque page du PDF en une image PIL. Le paramètre `dpi=300` garantit une résolution suffisante pour que le texte et les tracés restent lisibles lors de la détection YOLO et de l'OCR qui suivent.

> **Dépendance système :** `pdf2image` nécessite `poppler`.  
> Installation : `sudo apt install poppler-utils`

In [5]:
print('Conversion du PDF en images (300 DPI)...')
pages = convert_from_path(save_path, dpi=300)
print(f'✔  {len(pages)} page(s) trouvée(s).')

Conversion du PDF en images (300 DPI)...
✔  20 page(s) trouvée(s).


### 1.3 Détection YOLO et découpe des zones

Pour chaque page, YOLO prédit les boîtes englobantes avec un seuil de confiance de `0.212` (valeur basse car le modèle est spécialisé et les faux positifs sont rares).

Chaque zone détectée est découpée avec une **marge de 10 pixels** de chaque côté pour ne pas rogner les bords, puis sauvegardée en PNG. Le nom du fichier encode le numéro de page et la classe détectée.

In [6]:
for p_id, page in enumerate(pages):
    page_array = np.array(page)   # PIL → numpy RGB

    # Prédiction YOLO sur cette page
    result = model_extraction.predict(source=page, conf=0.212, verbose=False)

    for box in result[0].boxes:
        # Coordonnées de la boîte englobante
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        # Découpe avec marge de 10 px
        margin = 10
        crop = page_array[
            max(0, y1 - margin): min(page_array.shape[0], y2 + margin),
            max(0, x1 - margin): min(page_array.shape[1], x2 + margin)
        ]

        if crop is not None and crop.size > 0:
            cls      = result[0].names[int(box.cls[0])]
            pdf_name = Path(save_path).stem
            filename = f'{pdf_name}_{p_id}_{cls}.png'
            # Conversion RGB → BGR pour OpenCV
            cv2.imwrite(os.path.join(dir_path, filename),
                        cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))

print(f'✔  Traitement terminé. Images sauvegardées dans : {dir_path}')

✔  Traitement terminé. Images sauvegardées dans : parts/Lecture_des_composants/notebook/IMAGES


---
## 2. `merge.py` — Orchestration de la lecture des images

Ce script est une version améliorée de `prediction.py`. Au lieu de sauvegarder toutes les détections, il ne conserve que **la meilleure zone par classe** (celle avec la plus grande surface en pixels), applique des rotations si nécessaire, puis lance l'interface interactive de `test1.py`.

**Usage standalone :**
```bash
cd ~/GitHub/etudes-pratiques
python3 parts/Lecture_table/dev/merge.py notebook/notebook_files/Accuphase-E202amp.pdf models/extraction/best.pt
```

### 2.1 Imports et import de l'interface

### 2.2 Fonction principale `process_and_visualize()`

Toute la logique est encapsulée dans cette fonction. Elle reçoit le chemin du PDF et le chemin du modèle YOLO, et orchestre les 4 étapes suivantes.

#### Étape 1 & 2 : Chargement du modèle et conversion du PDF

Identique à `prediction.py` : le modèle YOLO est chargé, puis le PDF est converti page par page en images PIL à 300 DPI.

In [7]:
# Les imports suivants sont déjà chargés dans la section 1.
# Cette cellule est fournie à titre documentaire (fidèle à merge.py).
import os
import sys
import cv2
import numpy as np
from pdf2image import convert_from_path
from ultralytics import YOLO

# afficher() sera définie dans la section test1.py ci-dessous.
print('Imports merge.py OK')

Imports merge.py OK


#### Étape 2 : Sélection de la meilleure zone par classe

Le dictionnaire `best_images` stocke pour chaque classe un tuple `(image_découpée, surface_en_pixels)`. À chaque nouvelle détection, on calcule la surface (`largeur × hauteur`) et on ne garde la découpe que si elle est **plus grande** que celle déjà enregistrée. Ainsi, à la fin du parcours de toutes les pages, on a retenu la zone la plus grande pour chaque type.

#### Étape 3 : Rotations et sauvegarde temporaire

Les schémas et cartes sont souvent orientés en mode paysage dans les PDFs. On les redresse avant affichage :

- `Diagram` → rotation 90° **gauche** (counter-clockwise)
- `Board` → rotation 90° **droite** (clockwise)
- `Table` → aucune rotation

On effectue aussi la conversion **RGB → BGR**, nécessaire car PIL produit des images en RGB tandis qu'OpenCV (`cv2.imwrite`) attend du BGR.

#### Étape 4 : Lancement de l'interface

`Table` et `Diagram` sont obligatoires pour que l'interface ait du sens. `Board` est optionnel — `.get()` retourne `None` sans lever d'erreur si la clé est absente, et `afficher()` gère ce cas.

In [8]:
def process_and_visualize(pdf_path_arg, model_path):
    """
    Pipeline complet merge.py :
    1. Charge le modèle YOLO de zones
    2. Convertit le PDF en images PIL
    3. Détecte les zones et garde la meilleure surface par classe
    4. Applique les rotations (Diagram ←90°, Board →90°)
    5. Sauvegarde temporaire et appel à afficher()
    """
    try:
        model = YOLO(model_path)
    except Exception as e:
        print(f'Erreur chargement modèle : {e}')
        return

    # Étape 2 : Conversion PDF
    print('--- Conversion du PDF en images ---')
    pages = convert_from_path(pdf_path_arg, dpi=300)

    best_images = {'Table': (None, 0), 'Diagram': (None, 0), 'Board': (None, 0)}

    # Étape 2 : Extraction et sélection meilleure zone
    print('--- Extraction des zones via YOLO ---')
    for p_id, page in enumerate(pages):
        results      = model.predict(source=page, conf=0.212, verbose=False)
        page_array   = np.array(page)   # RGB

        for box in results[0].boxes:
            cls_name = results[0].names[int(box.cls[0])]

            # Mapping vers nos clés normalisées
            key = None
            if   cls_name.lower() == 'table':   key = 'Table'
            elif cls_name.lower() == 'diagram': key = 'Diagram'
            elif cls_name.lower() == 'board':   key = 'Board'

            if key:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                margin = 10
                crop = page_array[
                    max(0, y1-margin): min(page_array.shape[0], y2+margin),
                    max(0, x1-margin): min(page_array.shape[1], x2+margin)
                ]
                if crop is not None and crop.size > 0:
                    surface = crop.shape[0] * crop.shape[1]
                    if surface > best_images[key][1]:
                        best_images[key] = (crop, surface)

    # Étape 3 : Rotations et sauvegarde temporaire
    temp_paths = {}
    for cls in ['Table', 'Diagram', 'Board']:
        img_data, surf = best_images[cls]
        if img_data is not None:
            # Conversion RGB → BGR pour OpenCV
            img_to_save = cv2.cvtColor(img_data, cv2.COLOR_RGB2BGR)

            # Rotations selon la classe
            if cls == 'Diagram':
                print('Rotation du Schéma (90° gauche)...')
                img_to_save = cv2.rotate(img_to_save, cv2.ROTATE_90_COUNTERCLOCKWISE)
            elif cls == 'Board':
                print('Rotation de la Carte (90° droite)...')
                img_to_save = cv2.rotate(img_to_save, cv2.ROTATE_90_CLOCKWISE)

            path = f'parts/Lecture_des_composants/notebook/temp_{cls.lower()}.png'
            cv2.imwrite(path, img_to_save)
            temp_paths[cls] = path

    # Étape 4 : Lancement de l'interface
    if 'Table' in temp_paths and 'Diagram' in temp_paths:
        print('--- Affichage ---')
        afficher(
            temp_paths['Table'],
            temp_paths['Diagram'],
            temp_paths.get('Board')
        )
    else:
        print('\nERREUR : Table ou Diagram manquant.')
        print(f'Trouvés : {[k for k, v in best_images.items() if v[0] is not None]}')

print('✔  process_and_visualize() définie')

✔  process_and_visualize() définie


---
## 3. `test1.py` — Interface interactive de visualisation

Le but du script est de fournir une interface Matplotlib qui affiche côte à côte la table BOM, le schéma et la carte PCB, et permet de **cliquer sur un composant** (ex : `R12`) pour le localiser dans les trois images simultanément.

Il utilise deux moteurs de détection selon le type d'image :
- **Tesseract seul** pour la table BOM (texte structuré en colonnes)
- **YOLO + Tesseract** pour le schéma et la carte (étiquettes petites et dispersées)

**Usage standalone :**
```bash
cd ~/GitHub/etudes-pratiques
python3 parts/Lecture_table/dev/test1.py parts/Lecture_table/dev/IMAGES/table.png parts/Lecture_table/dev/IMAGES/schema.png parts/Lecture_table/dev/IMAGES/carte.png
```

### 3.1 Imports, configuration et constantes

Le script tente de forcer le backend graphique `TkAgg` pour Matplotlib (nécessaire sur certains systèmes Linux/Mac pour afficher une fenêtre interactive). Dans un notebook Jupyter, cette ligne est ignorée silencieusement et le backend par défaut est utilisé.

Le chargement du modèle `bestr.pt` est fait ici — si le fichier est absent, un avertissement est affiché (contrairement au script original qui appelle `sys.exit`, ce qui tuerait le kernel Jupyter).

`PREFIXES_VALIDES` liste les codes normalisés des composants électroniques : `R` = résistance, `C` = condensateur, `U` = circuit intégré, `D` = diode, `Q` = transistor, `L` = inductance, etc.

In [9]:
import cv2
import re
import numpy as np
import pytesseract
from ultralytics import YOLO
import matplotlib
try:
    matplotlib.use('TkAgg')
except Exception:
    pass   # backend non disponible en notebook → comportement par défaut
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors

# Modèle YOLO pour détecter les zones d'étiquettes sur schéma/carte
# bestr.pt = modèle entraîné sur les références composants (R, C, IC…)
# DIFFÉRENT de best.pt (qui détecte Table/Diagram/Board, pas les étiquettes)
PATH_TO_YOLO_MODEL = 'parts/Lecture_table/dev/bestr.pt'

try:
    yolo_model = YOLO(PATH_TO_YOLO_MODEL)
    print(f'✔  Modèle YOLO chargé : {PATH_TO_YOLO_MODEL}')
except Exception as e:
    yolo_model = None
    print(f'⚠️  Erreur chargement YOLO : {e}')
    print('   → detect_refs_hybrid() ne fonctionnera pas sans ce modèle.')

PREFIXES_VALIDES = ['R', 'C', 'U', 'D', 'Q', 'L', 'J', 'SW', 'F', 'TP', 'RV', 'Y', 'DZ', 'IC']

✔  Modèle YOLO chargé : parts/Lecture_table/dev/bestr.pt


### 3.2 `natural_sort_key()` — tri naturel des références

Un tri alphabétique classique produirait `R1, R10, R11, R2` (car `'10' < '2'` en comparaison de chaînes). Le tri naturel compare les parties numériques comme des entiers, donnant l'ordre correct : `R1, R2, R10, R11`.

La fonction découpe la chaîne en alternant segments texte et segments numériques, puis compare chaque segment dans son type propre.

In [10]:
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split(r'(\d+)', s)]

### 3.3 `clean_ocr_text()` — correction des erreurs OCR

L'OCR fait souvent des erreurs visuelles sur les petites polices des schémas. Cette fonction applique un pipeline de corrections en 4 étapes :

1. **Nettoyage** : ne garder que les caractères alphanumériques majuscules (supprimer ponctuation, espaces, caractères parasites)
2. **Extraction du préfixe** : chercher un préfixe valide dans le texte. Les préfixes sont triés par longueur décroissante pour matcher `SW` avant `S`, `IC` avant `I`, etc.
3. **Correction des confusions visuelles** : `S→5`, `A→4`, `O→0`, `I→1`, `L→1`, `Z→2`, `G→6`, `B→8`
4. **Suppression des zéros de remplissage** : `Q010 → Q10` (`.lstrip('0')`)

Si aucun préfixe valide n'est trouvé, la fonction retourne `None` (le mot est ignoré).

In [11]:
def clean_ocr_text(text):
    """
    Nettoyage strict :
    1. AR4 → R4  (suppression préfixes fantômes)
    2. Q010 → Q10 (suppression zéros de remplissage)
    3. RS → R5   (correction confusions visuelles)
    """
    t = re.sub(r'[^A-Z0-9]', '', text.upper())

    found_prefix   = None
    remaining_part = ''

    # Cherche le préfixe le plus long d'abord (SW avant S, IC avant I)
    for p in sorted(PREFIXES_VALIDES, key=len, reverse=True):
        match = re.search(r'(' + p + r')([A-Z0-9]+)', t)
        if match:
            found_prefix   = match.group(1)
            remaining_part = match.group(2)
            break

    if found_prefix:
        # Correction des confusions visuelles dans la partie numérique
        suffix = remaining_part.replace('S','5').replace('A','4').replace('O','0')
        suffix = suffix.replace('I','1').replace('L','1').replace('Z','2')
        suffix = suffix.replace('G','6').replace('B','8')

        # Extraction et nettoyage de la partie numérique
        num_match = re.search(r'\d+', suffix)
        if num_match:
            num_str = num_match.group().lstrip('0') or '0'
            return found_prefix + num_str

    return None

### 3.4 `preprocess_for_ocr()` — préparation d'une image pour Tesseract

Tesseract donne de bien meilleurs résultats sur des images :
- **en niveaux de gris** (pas de couleur à interpréter)
- **agrandies** (le texte des étiquettes est souvent très petit)
- **binarisées** (uniquement noir ou blanc, pas de nuances de gris)

La binarisation utilise la méthode d'**Otsu** : elle calcule automatiquement le seuil optimal qui sépare le fond du texte, sans paramètre à régler manuellement. Le paramètre `factor` contrôle le niveau d'agrandissement (3× par défaut, 4× pour les très petites étiquettes).

In [12]:
def preprocess_for_ocr(img, factor=3):
    """Prépare une petite zone de l'image pour Tesseract : niveaux de gris → agrandissement → Otsu."""
    if img.size == 0:
        return img
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=factor, fy=factor, interpolation=cv2.INTER_CUBIC)
    return cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]

### 3.5 `detect_refs_tesseract()` — analyse de la table BOM

La table BOM est un tableau structuré où les références de composants se trouvent dans la **colonne de gauche** (approximativement les 18 premiers % de la largeur). On n'analyse que cette bande pour éviter de lire les valeurs, descriptions ou quantités des autres colonnes.

`image_to_data` (avec `output_type=DICT`) est préféré à `image_to_string` car il retourne aussi les **coordonnées** de chaque mot détecté — indispensable pour savoir où zoomer ensuite dans l'interface.

Le mode `--psm 11` demande à Tesseract de chercher des mots épars (pas de mise en page structurée supposée), adapté aux colonnes de références.

In [13]:
def detect_refs_tesseract(img):
    """Analyse de la table BOM — colonne de gauche (18% de la largeur)."""
    if img is None:
        return []
    h, w  = img.shape[:2]
    crop  = img[:, :int(w * 0.18)]   # seulement la colonne gauche
    proc  = preprocess_for_ocr(crop, factor=2)
    data  = pytesseract.image_to_data(
        proc, config='--oem 3 --psm 11',
        output_type=pytesseract.Output.DICT
    )

    found_refs  = []
    last_prefix = 'R'

    for i, txt in enumerate(data['text']):
        clean = txt.strip().upper()
        if len(clean) < 2:
            continue

        # Certaines cellules contiennent plusieurs refs séparées par des virgules : "R12, R13"
        parts = re.split(r',', clean)
        for p in parts:
            m = re.match(r'([A-Z]+)(\d+)', p)
            if m:
                last_prefix = m.group(1)
                found_refs.append({
                    'ref': last_prefix + m.group(2),
                    'cx':  data['left'][i] / 2,
                    'cy':  data['top'][i]  / 2,
                    'bbox': (
                        data['left'][i] / 2,
                        data['top'][i]  / 2,
                        (data['left'][i] + data['width'][i])  / 2,
                        (data['top'][i]  + data['height'][i]) / 2
                    )
                })
            else:
                # Juste un nombre → on réutilise le dernier préfixe connu
                num = re.search(r'\d+', p)
                if num:
                    found_refs.append({
                        'ref': last_prefix + num.group(),
                        'cx':  data['left'][i] / 2,
                        'cy':  data['top'][i]  / 2,
                        'bbox': (
                            data['left'][i] / 2,
                            data['top'][i]  / 2,
                            (data['left'][i] + data['width'][i])  / 2,
                            (data['top'][i]  + data['height'][i]) / 2
                        )
                    })
    return found_refs

### 3.6 `detect_refs_hybrid()` — analyse du schéma et de la carte

Pour les schémas et cartes PCB, les étiquettes de composants sont petites, orientées dans tous les sens, et dispersées sur toute l'image. Une approche OCR globale produirait trop de bruit. On adopte donc une stratégie en deux temps :

1. **YOLO `bestr.pt`** localise les zones de l'image qui contiennent probablement une étiquette (`imgsz=1280` : taille d'entrée agrandie pour mieux détecter les très petits éléments)
2. Pour chaque zone détectée, **Tesseract** lit le texte en mode `--psm 7` (ligne de texte unique) avec une liste blanche de caractères pour éviter les faux positifs

Le résultat est passé dans `clean_ocr_text()` pour corriger les erreurs visuelles courantes.

In [14]:
def detect_refs_hybrid(img):
    """Analyse du schéma/carte : YOLO (localise) + Tesseract PSM 7 (lit)."""
    if img is None or yolo_model is None:
        return []

    results    = yolo_model.predict(source=img, conf=0.20, imgsz=1280, verbose=False)
    found_refs = []

    for result in results:
        for box in result.boxes:
            b   = box.xyxy[0].cpu().numpy().astype(int)
            # Découpe la zone avec un petit buffer de 5px
            roi = img[max(0, b[1]-5): b[3]+5,
                      max(0, b[0]-5): b[2]+5]
            if roi.size > 0:
                roi_proc = preprocess_for_ocr(roi, factor=4)
                config   = '--psm 7 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
                raw_text = pytesseract.image_to_string(roi_proc, config=config).strip()

                final_ref = clean_ocr_text(raw_text)
                if final_ref:
                    found_refs.append({
                        'ref':  final_ref,
                        'cx':   float((b[0] + b[2]) / 2),
                        'cy':   float((b[1] + b[3]) / 2),
                        'bbox': (float(b[0]), float(b[1]), float(b[2]), float(b[3]))
                    })
    return found_refs

### 3.7 `afficher()` — construction de l'interface Matplotlib

C'est le point d'entrée de l'interface. Elle charge les images, lance les détections sur chacune, puis construit la figure Matplotlib avec 3 ou 4 panneaux selon si la carte est fournie.

Les résultats des détections sont stockés dans des dictionnaires `{référence: infos}` (ex : `{"R12": {"cx": 450, "cy": 230, "bbox": (...)}}`). Cela permet une recherche en O(1) lors des interactions.

### 3.8 Panneau de liste cliquable

Les composants de la table BOM sont affichés sous forme de texte Matplotlib dans le panneau de gauche. Le paramètre `picker=5` rend chaque texte cliquable (zone de 5 px autour du texte). Les composants sont groupés par préfixe et colorés par groupe (10 couleurs distinctes issues de `TABLEAU_COLORS`).

`text_mapping` associe chaque objet texte Matplotlib à son identifiant de composant — indispensable pour savoir sur quel composant l'utilisateur a cliqué dans le callback `pick_event`.

### 3.9 Rectangles de focus et fonction `do_focus()`

Un rectangle `patches.Rectangle` invisible est pré-créé pour chaque panneau image. Quand `do_focus()` est appelée avec un identifiant de composant, elle :
1. Cherche le composant dans chaque dictionnaire
2. Si trouvé : zoome le panneau autour de la position du composant (`set_xlim` / `set_ylim`), repositionne et rend visible le rectangle rouge
3. Si absent de ce panneau : masque le rectangle

Note : en Matplotlib, l'axe Y est inversé par rapport aux coordonnées image (origine en haut), d'où `set_ylim(cy + zoom, cy - zoom)` pour zoomer correctement.

### 3.10 Gestion des événements souris

Trois types d'interactions sont disponibles dans l'interface :

| Interaction | Effet |
|---|---|
| Clic gauche sur la liste | Zoom sur le composant dans les 3 images + rectangle rouge |
| Clic gauche sur une image | Trouve le composant le plus proche du clic (distance euclidienne < 200 px) |
| Clic droit n'importe où | Réinitialise toutes les vues à leur niveau de zoom original |
| Molette sur une image | Zoom libre centré sur la position du curseur (×1,5 par cran) |

La recherche du composant le plus proche utilise `np.linalg.norm` pour calculer la distance euclidienne entre le point cliqué et tous les centres de composants.

### 3.11 `on_scroll()` — zoom à la molette

Le zoom est implémenté en redimensionnant les limites de l'axe autour du point pointé par le curseur. Un facteur 1,5 est appliqué à chaque cran de molette. Scroll vers le haut = zoom avant (division par 1,5), scroll vers le bas = zoom arrière (multiplication par 1,5).

In [15]:
def afficher(table_path, schema_path, carte_path=None):
    """
    Interface interactive Matplotlib 3 ou 4 panneaux.
    - Panneau gauche : liste cliquable des refs BOM
    - Panneaux droite : table, schéma, carte (optionnel)
    Interactions : clic liste → zoom, clic image → ref proche, clic droit → reset, molette → zoom
    """
    img_t = cv2.imread(table_path)
    img_s = cv2.imread(schema_path)
    img_c = cv2.imread(carte_path) if carte_path else None

    print('Analyse en cours...')
    dict_table  = {r['ref']: r for r in detect_refs_tesseract(img_t)}
    dict_schema = {r['ref']: r for r in detect_refs_hybrid(img_s)}
    dict_carte  = {r['ref']: r for r in detect_refs_hybrid(img_c)} if img_c is not None else {}
    print(f' Bilan : Table({len(dict_table)}) | Schéma({len(dict_schema)}) | Carte({len(dict_carte)})')

    # Construction de la figure
    fig  = plt.figure(figsize=(18, 9))
    cols = 4 if img_c is not None else 3
    gs   = fig.add_gridspec(1, cols, width_ratios=[1.2, 2, 2, 2][:cols])

    ax_list   = fig.add_subplot(gs[0, 0])
    ax_table  = fig.add_subplot(gs[0, 1])
    ax_schema = fig.add_subplot(gs[0, 2])
    ax_carte  = fig.add_subplot(gs[0, 3]) if img_c is not None else None
    axes_images = [ax for ax in [ax_table, ax_schema, ax_carte] if ax]

    ax_table.imshow(cv2.cvtColor(img_t, cv2.COLOR_BGR2RGB))
    ax_schema.imshow(cv2.cvtColor(img_s, cv2.COLOR_BGR2RGB))
    if ax_carte:
        ax_carte.imshow(cv2.cvtColor(img_c, cv2.COLOR_BGR2RGB))
    for ax in [ax_list] + axes_images:
        ax.axis('off')

    # Panneau de liste cliquable (§3.8)
    ids          = sorted(dict_table.keys(), key=natural_sort_key)
    colors       = list(mcolors.TABLEAU_COLORS.values())
    text_mapping = {}
    y_pos, last_prefix, color_idx = 0.98, None, 0

    for ident in ids:
        m      = re.match(r'([A-Z]+)', ident)
        prefix = m.group(1) if m else '?'
        if prefix != last_prefix:
            if last_prefix:
                y_pos -= 0.02
            ax_list.text(0.05, y_pos, f'--- {prefix} ---',
                         fontsize=10, fontweight='bold', alpha=0.7)
            y_pos    -= 0.03
            color_idx = (color_idx + 1) % len(colors)

        txt_obj = ax_list.text(0.15, y_pos, ident,
                               fontsize=9, picker=5, fontweight='bold',
                               color=colors[color_idx])
        text_mapping[txt_obj] = ident
        last_prefix = prefix
        y_pos      -= 0.025
        if y_pos < 0.02:
            break

    # Rectangles de focus (§3.9)
    rects = {
        ax: patches.Rectangle((0, 0), 0, 0,
                               edgecolor='red', facecolor='none',
                               linewidth=2, visible=False)
        for ax in axes_images
    }
    for ax in axes_images:
        ax.add_patch(rects[ax])
    orig_lims = {ax: (ax.get_xlim(), ax.get_ylim()) for ax in axes_images}

    def do_focus(ident):
        zoom  = 350
        found = False
        for ax, dic in [(ax_table, dict_table), (ax_schema, dict_schema), (ax_carte, dict_carte)]:
            if ax and ident in dic:
                info = dic[ident]
                ax.set_xlim(info['cx'] - zoom, info['cx'] + zoom)
                ax.set_ylim(info['cy'] + zoom, info['cy'] - zoom)
                b = info['bbox']
                rects[ax].set_xy((b[0], b[1]))
                rects[ax].set_width(b[2]  - b[0])
                rects[ax].set_height(b[3] - b[1])
                rects[ax].set_visible(True)
                found = True
            elif ax:
                rects[ax].set_visible(False)
        if found:
            fig.suptitle(f'Focus : {ident}', color='red', fontsize=16, fontweight='bold')
            plt.draw()

    # Gestion des événements (§3.10)
    def on_click(event):
        if event.button == 3:   # clic droit → reset
            for ax in axes_images:
                ax.set_xlim(orig_lims[ax][0])
                ax.set_ylim(orig_lims[ax][1])
                rects[ax].set_visible(False)
            fig.suptitle('')
            plt.draw()
        elif event.inaxes in axes_images and event.button == 1:
            curr_dict = {ax_table: dict_table,
                         ax_schema: dict_schema,
                         ax_carte:  dict_carte}.get(event.inaxes)
            if curr_dict:
                pts = np.array([[r['cx'], r['cy']] for r in curr_dict.values()])
                if len(pts) > 0:
                    dist = np.linalg.norm(pts - [event.xdata, event.ydata], axis=1)
                    idx  = np.argmin(dist)
                    if dist[idx] < 200:
                        do_focus(list(curr_dict.keys())[idx])

    fig.canvas.mpl_connect('pick_event',         lambda e: do_focus(text_mapping[e.artist]))
    fig.canvas.mpl_connect('button_press_event', on_click)
    fig.canvas.mpl_connect('scroll_event',       lambda e: on_scroll(e, axes_images))

    print('Interface prête.')
    plt.tight_layout()
    plt.show()


def on_scroll(event, axes_images):
    """Zoom à la molette centré sur la position du curseur (§3.11)."""
    if event.inaxes not in axes_images:
        return
    ax    = event.inaxes
    scale = 1 / 1.5 if event.button == 'up' else 1.5
    cur_xlim, cur_ylim = ax.get_xlim(), ax.get_ylim()
    new_w = (cur_xlim[1] - cur_xlim[0]) * scale
    new_h = (cur_ylim[1] - cur_ylim[0]) * scale
    ax.set_xlim([event.xdata - new_w / 2, event.xdata + new_w / 2])
    ax.set_ylim([event.ydata + new_h / 2, event.ydata - new_h / 2])
    plt.draw()

print('✔  afficher() et on_scroll() définies')

✔  afficher() et on_scroll() définies


---
## 4. `extract_easyocr.py` — OCR avancé des étiquettes composants

Cette section s'insère **après le merge** (étape 2) et **avant l'interface** (étape 3).  
Une fois que `process_and_visualize()` a sélectionné les meilleures images (Diagram, Board),  
le pipeline OCR v7 est appliqué pour lire les étiquettes avec une précision maximale.

**Pipeline OCR v7 :**
1. Auto-détection des paramètres selon la résolution (DPI estimé, contraste, fond)
2. YOLO `bestr.pt` localise les zones d'étiquettes
3. **Merge des boîtes voisines** — fusionne `"R"` + `"516"` en une seule boîte → OCR plus fiable
4. **EasyOCR** lit chaque zone (3 tentatives : CLAHE → rotation 90° → Otsu)
5. **Correcteur OCR** scan linéaire + dictionnaire de corrections post-OCR
6. **Déduplication** — même ref détectée plusieurs fois → on garde le meilleur score

> ℹ️ EasyOCR est plus robuste qu'un Tesseract PSM 7 sur les petites polices techniques des PCB.

In [16]:
import time
import easyocr

print('⏳  Chargement EasyOCR (langues latines)…', end='', flush=True)
_t0 = time.time()
# Langues à script latin — partagent le même modèle sous-jacent dans EasyOCR,
# donc pas de surcoût mémoire significatif. Améliore la robustesse sur les
# polices techniques variées des schémas électroniques.
READER = easyocr.Reader(
    ['en', 'fr', 'de', 'es', 'it', 'nl', 'pt'],
    gpu=True, verbose=False
)
print(f'  ✔  EasyOCR prêt  ({time.time()-_t0:.1f}s)')

⏳  Chargement EasyOCR (langues latines)…  ✔  EasyOCR prêt  (2.1s)


### Correcteur OCR, merge de boîtes, preprocessing et pipeline

Ces fonctions constituent le cœur de `extract_easyocr.py` v7. Elles sont reproduites ici directement pour que le notebook soit autosuffisant.

In [17]:
import re as _re
import json
import csv
from datetime import datetime

# ── Constantes OCR ────────────────────────────────────────────────────────────
ALLOWLIST    = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789-_'
OCR_CONF_MIN = 0.15

# ── Correcteur de confusions OCR ──────────────────────────────────────────────
PREFIX_FIXES = {'0': 'O', '1': 'I', '8': 'B', '6': 'G', '5': 'S'}
DIGIT_FIXES  = {
    'I': '1', 'J': '1', 'L': '1', '|': '1',
    'O': '0', 'Q': '0', 'D': '0',
    'G': '6', 'Z': '2', 'S': '5', 'B': '8', 'T': '7',
}

# Corrections post-OCR : erreurs systématiques observées sur ce type de schéma
POST_CORRECTIONS = {
    'UC402': 'IC402', 'UC405': 'IC405',
    'RSI9':  'R519',  'RL26':  'R126',
    'S501':  'C501',  'B178':  'BC178',
    'R62I':  'R621',  'C41J':  'C411',
    'C50G':  'C506',  'BCI78': 'BC178',
}

# Regex de validation des références composants
COMPONENT_PATTERN = _re.compile(
    r'^('
    r'[A-Z]{1,2}\s?\d{3,5}[A-Z]?'
    r'|[A-Z]{1,3}\d{2,5}[A-Z]?'
    r'|[A-Z]{2,4}\d{1,4}[A-Z]?'
    r'|St\s?\d{3}'
    r'|[A-Z]{1,3}\d+[-_]\d+'
    r')$',
    _re.IGNORECASE
)


def fix_ocr_confusion(text: str) -> str:
    """
    Scan linéaire : trouve le 1er vrai chiffre comme frontière préfixe / numérique.
    - Avant : corrige chiffres mal lus → lettres  (0→O, 1→I…)
    - Après : corrige lettres mal lues → chiffres (I→1, O→0, G→6…)
    - Dernier char lettre seule = suffixe, conservé tel quel
    Exemples : "C41J"→"C411", "R51G"→"R516", "R516A"→"R516A"
    """
    if not text:
        return text
    text = text.strip().upper().replace(' ', '')
    i = 0
    while i < len(text) and not text[i].isdigit():
        i += 1
    if i == len(text):
        return text
    prefix_raw, rest = text[:i], text[i:]
    if len(rest) > 1 and rest[-1].isalpha():
        suffix, digits_raw = rest[-1], rest[:-1]
    else:
        suffix, digits_raw = '', rest
    fixed_prefix = ''.join(PREFIX_FIXES.get(c, c) if c.isdigit() else c for c in prefix_raw)
    fixed_digits = ''.join(c if c.isdigit() else DIGIT_FIXES.get(c, c) for c in digits_raw)
    return fixed_prefix + fixed_digits + suffix


def apply_post_corrections(text: str) -> str:
    key = text.strip().upper().replace(' ', '')
    return POST_CORRECTIONS.get(key, text)


def is_valid_ref(text: str) -> bool:
    return bool(COMPONENT_PATTERN.match(text.strip()))


# ── Merge de boîtes voisines ──────────────────────────────────────────────────
def merge_nearby_boxes(boxes, gap_ratio=0.8, overlap_y_ratio=0.4):
    """
    Fusionne les paires de boîtes qui sont probablement 2 morceaux du même label.
    Critères : alignées verticalement + gap horizontal ≤ gap_ratio × hauteur moyenne.
    """
    if not boxes:
        return boxes
    boxes = sorted(boxes, key=lambda b: b[0])
    merged, changed = list(boxes), True
    while changed:
        changed, new_merged, used = False, [], [False] * len(merged)
        for i in range(len(merged)):
            if used[i]: continue
            x1a, y1a, x2a, y2a, ca = merged[i]
            best_j, best_gap = -1, float('inf')
            for j in range(i + 1, len(merged)):
                if used[j]: continue
                x1b, y1b, x2b, y2b, cb = merged[j]
                if x1b < x2a: continue
                h_avg = ((y2a - y1a) + (y2b - y1b)) / 2
                gap   = x1b - x2a
                if gap > gap_ratio * h_avg: continue
                overlap_y = min(y2a, y2b) - max(y1a, y1b)
                min_h = min(y2a - y1a, y2b - y1b)
                if min_h == 0 or overlap_y / min_h < overlap_y_ratio: continue
                if gap < best_gap: best_gap, best_j = gap, j
            if best_j >= 0:
                x1b, y1b, x2b, y2b, cb = merged[best_j]
                new_merged.append((min(x1a,x1b), min(y1a,y1b),
                                   max(x2a,x2b), max(y2a,y2b), (ca+cb)/2))
                used[i] = used[best_j] = True
                changed = True
            else:
                new_merged.append(merged[i])
                used[i] = True
        merged = new_merged
    return merged


# ── Preprocessing des crops ───────────────────────────────────────────────────
def preprocess_crop(crop_bgr, scale=5):
    """Upscale × scale + sharpening + CLAHE + border blanc."""
    h, w  = crop_bgr.shape[:2]
    big   = cv2.resize(crop_bgr, (w*scale, h*scale), interpolation=cv2.INTER_CUBIC)
    sharp = cv2.filter2D(big, -1, np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]))
    gray  = cv2.cvtColor(sharp, cv2.COLOR_BGR2GRAY)
    enhanced = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(gray)
    bgr   = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)
    return cv2.copyMakeBorder(bgr, 10,10,10,10, cv2.BORDER_CONSTANT, value=(255,255,255))


def preprocess_crop_otsu(crop_bgr, scale=5):
    """Variante binarisation Otsu — meilleure sur fond sombre ou contraste inversé."""
    h, w = crop_bgr.shape[:2]
    big  = cv2.resize(crop_bgr, (w*scale, h*scale), interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(big, cv2.COLOR_BGR2GRAY)
    _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    bgr  = cv2.cvtColor(bw, cv2.COLOR_GRAY2BGR)
    return cv2.copyMakeBorder(bgr, 10,10,10,10, cv2.BORDER_CONSTANT, value=(255,255,255))


# ── Auto-détection des paramètres ────────────────────────────────────────────
def auto_params(img):
    """
    Analyse l'image et retourne les paramètres optimaux automatiquement :
    - imgsz YOLO et scale crop selon résolution en pixels
    - conf YOLO selon le contraste (std de l'image grise)
    - invert=True si le fond est sombre (mean < 80)
    """
    h, w  = img.shape[:2]
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mean  = float(gray.mean())
    std   = float(gray.std())
    px    = w * h
    if   px > 3_000_000: imgsz, scale = 1280, 5
    elif px > 1_000_000: imgsz, scale = 1024, 6
    elif px > 300_000:   imgsz, scale = 640,  8
    else:                imgsz, scale = 640,  10
    conf   = 0.25 if std < 25 else (0.30 if std < 45 else 0.40)
    invert = mean < 80
    print(f'  AUTO  {w}×{h}px | contraste={std:.0f} | luminosité={mean:.0f} '
          f'| imgsz={imgsz} | scale={scale} | conf={conf}'
          + ('  [INVERSION fond sombre]' if invert else ''))
    return {'imgsz': imgsz, 'scale': scale, 'conf': conf, 'invert': invert}


# ── EasyOCR runner ────────────────────────────────────────────────────────────
def _run_easyocr(img_bgr):
    """Lance EasyOCR avec allowlist et retourne la liste (text, conf) triée par confiance."""
    try:
        results = READER.readtext(img_bgr, detail=1, paragraph=False,
                                  allowlist=ALLOWLIST, width_ths=0.7)
    except Exception:
        return []
    out = []
    for (_, text, conf) in results:
        if conf < OCR_CONF_MIN: continue
        t = _re.sub(r'\s+', '', text.strip().upper())
        if t: out.append((t, conf))
    return sorted(out, key=lambda x: x[1], reverse=True)


def run_ocr(crop_bgr, scale=5):
    """
    Séquence de 3 tentatives, arrêt dès qu'on trouve une ref valide :
      1. CLAHE normal
      2. CLAHE + rotation 90° (si crop vertical ou rien trouvé)
      3. Otsu (fallback fond sombre / contraste inversé)
    Retourne (text_corrigé, is_valid, conf).
    """
    def best_from(img_prep):
        hits = _run_easyocr(img_prep)
        if not hits: return '', False, 0.0
        for (t, c) in hits:
            fixed = apply_post_corrections(fix_ocr_confusion(t))
            if is_valid_ref(fixed): return fixed, True, c
        fixed = apply_post_corrections(fix_ocr_confusion(hits[0][0]))
        return fixed, False, hits[0][1]

    pre_clahe = preprocess_crop(crop_bgr, scale=scale)
    h, w = pre_clahe.shape[:2]
    text, valid, conf = best_from(pre_clahe)
    if valid: return text, valid, conf
    if h > w * 1.3 or not text:
        t2, v2, c2 = best_from(cv2.rotate(pre_clahe, cv2.ROTATE_90_CLOCKWISE))
        if v2 or (t2 and c2 > conf): text, valid, conf = t2, v2, c2
    if valid: return text, valid, conf
    t3, v3, c3 = best_from(preprocess_crop_otsu(crop_bgr, scale=scale))
    if v3 or (t3 and c3 > conf): return t3, v3, c3
    return text, valid, conf


# ── Déduplication finale des refs ────────────────────────────────────────────
def deduplicate_refs(detections):
    """
    Si la même ref valide apparaît plusieurs fois, on garde uniquement
    la détection avec le meilleur score (yolo_conf × ocr_conf).
    Les refs invalides ne sont pas touchées.
    """
    seen, result = {}, []
    for d in detections:
        if not d['is_valid_ref']:
            result.append(d); continue
        key   = d['text'].replace(' ', '').upper()
        score = d['yolo_conf'] * d['ocr_conf']
        if key not in seen:
            seen[key] = len(result)
            result.append(d)
        else:
            ex = result[seen[key]]
            if score > ex['yolo_conf'] * ex['ocr_conf']:
                result[seen[key]] = d
    return result


print('✔  Fonctions OCR v7 définies')

✔  Fonctions OCR v7 définies


### Fonction principale : `run_ocr_on_image()`

Applique le pipeline OCR v7 complet sur une image (Diagram ou Board) :
1. Charge l'image et détecte les paramètres automatiquement
2. Lance YOLO pour localiser les zones d'étiquettes
3. Fusionne les boîtes voisines (merge)
4. Lance `run_ocr()` sur chaque crop
5. Déduplique
6. Sauvegarde PNG annoté, JSON et CSV
7. Retourne un dict `{ref: {cx, cy, bbox}}` compatible avec `afficher()`

In [18]:
def run_ocr_on_image(img_path, yolo_model_ocr, output_dir='parts/Lecture_des_composants/notebook/results', label=None):
    """
    Pipeline OCR v7 complet sur une image (Diagram ou Board).

    Paramètres
    ----------
    img_path      : str  — chemin vers l'image produite par process_and_visualize
    yolo_model_ocr: YOLO — modèle pour détecter les étiquettes
    output_dir    : str  — dossier de sortie CSV / JSON / image annotée
    label         : str  — nom court pour les fichiers (ex: 'diagram')

    Retourne
    --------
    dict : {ref: {'cx','cy','bbox'}} compatible avec afficher()
    """
    if yolo_model_ocr is None:
        print(f'\u26a0\ufe0f  Modèle YOLO absent — OCR impossible sur {label or str(img_path)}')
        return {}
    os.makedirs(output_dir, exist_ok=True)
    label = label or Path(img_path).stem

    img = cv2.imread(str(img_path))
    if img is None:
        print(f'⚠️  Image introuvable : {img_path}')
        return {}

    # Auto-paramètres
    params = auto_params(img)
    if params['invert']:
        img = cv2.bitwise_not(img)

    # YOLO
    print(f'  YOLO sur {label}…', end='', flush=True)
    yolo_result = yolo_model_ocr(img, conf=params['conf'],
                                 imgsz=params['imgsz'], verbose=False)[0]
    boxes_raw = [
        (int(b.xyxy[0][0]), int(b.xyxy[0][1]),
         int(b.xyxy[0][2]), int(b.xyxy[0][3]), float(b.conf[0]))
        for b in yolo_result.boxes
    ]
    boxes = merge_nearby_boxes(boxes_raw)
    print(f'  {len(boxes_raw)} zones → {len(boxes)} après merge')

    # OCR
    detections, PAD = [], 4
    for i, (x1, y1, x2, y2, conf_yolo) in enumerate(boxes):
        crop = img[max(0,y1-PAD): min(img.shape[0],y2+PAD),
                   max(0,x1-PAD): min(img.shape[1],x2+PAD)]
        if crop.size == 0: continue
        text, valid, ocr_conf = run_ocr(crop, scale=params['scale'])
        detections.append({
            'id': i+1, 'text': text, 'is_valid_ref': valid,
            'yolo_conf': round(conf_yolo, 3),
            'ocr_conf':  round(ocr_conf, 3),
            'score':     round(conf_yolo * ocr_conf, 3),
            'bbox':   {'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2},
            'center': {'x': (x1+x2)//2, 'y': (y1+y2)//2},
        })
        sys.stdout.write(f'\r  OCR {i+1}/{len(boxes)}')
        sys.stdout.flush()
    sys.stdout.write('\n')

    detections = deduplicate_refs(detections)
    valid_dets  = [d for d in detections if d['is_valid_ref']]
    print(f'  ✔  {len(valid_dets)} refs valides')

    # Exports
    ts       = datetime.now().strftime('%Y%m%d_%H%M%S')
    ann_path = os.path.join(output_dir, f'{label}_{ts}_annotated.png')

    # Image annotée
    out_img = img.copy()
    for d in detections:
        b     = d['bbox']
        color = (0, 200, 0) if d['is_valid_ref'] else (0, 140, 255)
        cv2.rectangle(out_img, (b['x1'], b['y1']), (b['x2'], b['y2']), color, 2)
        if d['text']:
            cv2.putText(out_img, d['text'], (b['x1']+2, b['y1']-3),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,255), 1)
    cv2.imwrite(ann_path, out_img)

    # JSON
    json_path = os.path.join(output_dir, f'{label}_{ts}_results.json')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(detections, f, ensure_ascii=False, indent=2)

    # CSV
    csv_path = os.path.join(output_dir, f'{label}_{ts}_results.csv')
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=['id','text','is_valid_ref','yolo_conf','ocr_conf',
                                          'score','x1','y1','x2','y2','cx','cy'])
        w.writeheader()
        for d in detections:
            w.writerow({'id': d['id'], 'text': d['text'], 'is_valid_ref': d['is_valid_ref'],
                        'yolo_conf': d['yolo_conf'], 'ocr_conf': d['ocr_conf'],
                        'score': d['score'],
                        'x1': d['bbox']['x1'], 'y1': d['bbox']['y1'],
                        'x2': d['bbox']['x2'], 'y2': d['bbox']['y2'],
                        'cx': d['center']['x'], 'cy': d['center']['y']})

    print(f'  Fichiers : {ann_path}')

    # Retourne le dict compatible afficher()
    return {
        d['text']: {
            'cx':   d['center']['x'],
            'cy':   d['center']['y'],
            'bbox': (d['bbox']['x1'], d['bbox']['y1'],
                     d['bbox']['x2'], d['bbox']['y2'])
        }
        for d in valid_dets
    }

print('✔  run_ocr_on_image() définie')

✔  run_ocr_on_image() définie


### `afficher()` — version mise à jour (support OCR EasyOCR)

La signature est étendue avec deux paramètres optionnels `ocr_schema` et `ocr_carte`.  
Si fournis, ils **remplacent** `detect_refs_hybrid()` par les résultats OCR v7 (plus précis).  
Sans ces paramètres, le comportement est **identique à l'original** (fallback Tesseract).

In [19]:
def afficher_v2(table_path, schema_path, carte_path=None,
                ocr_schema=None, ocr_carte=None):
    """
    Interface interactive Matplotlib — version étendue OCR v7.

    Paramètres
    ----------
    table_path, schema_path, carte_path : chemins des images
    ocr_schema : dict optionnel {ref: {cx,cy,bbox}} issu de run_ocr_on_image()
    ocr_carte  : dict optionnel {ref: {cx,cy,bbox}} issu de run_ocr_on_image()

    Si ocr_schema/ocr_carte sont None → fallback detect_refs_hybrid() (comportement original).
    """
    import matplotlib.patches as _patches
    import matplotlib.colors  as _mcolors

    img_t = cv2.imread(table_path)
    img_s = cv2.imread(schema_path)
    img_c = cv2.imread(carte_path) if carte_path else None

    print('Analyse en cours...')
    dict_table  = {r['ref']: r for r in detect_refs_tesseract(img_t)}

    # Schéma : OCR v7 si fourni, sinon fallback Tesseract
    if ocr_schema is not None:
        dict_schema = {
            ref: {'cx': info['cx'], 'cy': info['cy'],
                  'ref': ref, 'bbox': info['bbox']}
            for ref, info in ocr_schema.items()
        }
    else:
        dict_schema = {r['ref']: r for r in detect_refs_hybrid(img_s)}

    # Carte : OCR v7 si fourni, sinon fallback Tesseract
    if img_c is not None:
        if ocr_carte is not None:
            dict_carte = {
                ref: {'cx': info['cx'], 'cy': info['cy'],
                      'ref': ref, 'bbox': info['bbox']}
                for ref, info in ocr_carte.items()
            }
        else:
            dict_carte = {r['ref']: r for r in detect_refs_hybrid(img_c)}
    else:
        dict_carte = {}

    print(f' Bilan : Table({len(dict_table)}) | Schéma({len(dict_schema)}) | Carte({len(dict_carte)})')

    # Construction figure (identique à afficher())
    fig  = plt.figure(figsize=(18, 9))
    cols = 4 if img_c is not None else 3
    gs   = fig.add_gridspec(1, cols, width_ratios=[1.2, 2, 2, 2][:cols])
    ax_list   = fig.add_subplot(gs[0, 0])
    ax_table  = fig.add_subplot(gs[0, 1])
    ax_schema = fig.add_subplot(gs[0, 2])
    ax_carte  = fig.add_subplot(gs[0, 3]) if img_c is not None else None
    axes_images = [ax for ax in [ax_table, ax_schema, ax_carte] if ax]

    ax_table.imshow(cv2.cvtColor(img_t, cv2.COLOR_BGR2RGB))
    ax_schema.imshow(cv2.cvtColor(img_s, cv2.COLOR_BGR2RGB))
    if ax_carte: ax_carte.imshow(cv2.cvtColor(img_c, cv2.COLOR_BGR2RGB))
    for ax in [ax_list] + axes_images: ax.axis('off')

    ids          = sorted(dict_table.keys(), key=natural_sort_key)
    colors       = list(_mcolors.TABLEAU_COLORS.values())
    text_mapping = {}
    y_pos, last_prefix, color_idx = 0.98, None, 0
    for ident in ids:
        m = _re.match(r'([A-Z]+)', ident)
        prefix = m.group(1) if m else '?'
        if prefix != last_prefix:
            if last_prefix: y_pos -= 0.02
            ax_list.text(0.05, y_pos, f'--- {prefix} ---', fontsize=10, fontweight='bold', alpha=0.7)
            y_pos    -= 0.03
            color_idx = (color_idx + 1) % len(colors)
        txt_obj = ax_list.text(0.15, y_pos, ident, fontsize=9, picker=5,
                               fontweight='bold', color=colors[color_idx])
        text_mapping[txt_obj] = ident
        last_prefix = prefix
        y_pos -= 0.025
        if y_pos < 0.02: break

    rects = {
        ax: _patches.Rectangle((0,0),0,0, edgecolor='red', facecolor='none',
                                linewidth=2, visible=False)
        for ax in axes_images
    }
    for ax in axes_images: ax.add_patch(rects[ax])
    orig_lims = {ax: (ax.get_xlim(), ax.get_ylim()) for ax in axes_images}

    def do_focus(ident):
        zoom = 350; found = False
        for ax, dic in [(ax_table, dict_table), (ax_schema, dict_schema), (ax_carte, dict_carte)]:
            if ax and ident in dic:
                info = dic[ident]
                ax.set_xlim(info['cx']-zoom, info['cx']+zoom)
                ax.set_ylim(info['cy']+zoom, info['cy']-zoom)
                b = info['bbox']
                rects[ax].set_xy((b[0], b[1]))
                rects[ax].set_width(b[2]-b[0]); rects[ax].set_height(b[3]-b[1])
                rects[ax].set_visible(True); found = True
            elif ax: rects[ax].set_visible(False)
        if found: fig.suptitle(f'Focus : {ident}', color='red', fontsize=16, fontweight='bold'); plt.draw()

    def on_click_v2(event):
        if event.button == 3:
            for ax in axes_images:
                ax.set_xlim(orig_lims[ax][0]); ax.set_ylim(orig_lims[ax][1]); rects[ax].set_visible(False)
            fig.suptitle(''); plt.draw()
        elif event.inaxes in axes_images and event.button == 1:
            curr = {ax_table: dict_table, ax_schema: dict_schema, ax_carte: dict_carte}.get(event.inaxes)
            if curr:
                pts = np.array([[r['cx'], r['cy']] for r in curr.values()])
                if len(pts) > 0:
                    dist = np.linalg.norm(pts - [event.xdata, event.ydata], axis=1)
                    idx  = np.argmin(dist)
                    if dist[idx] < 200: do_focus(list(curr.keys())[idx])

    fig.canvas.mpl_connect('pick_event',         lambda e: do_focus(text_mapping[e.artist]))
    fig.canvas.mpl_connect('button_press_event', on_click_v2)
    fig.canvas.mpl_connect('scroll_event',       lambda e: on_scroll(e, axes_images))
    print('Interface prête (OCR v7).')
    plt.tight_layout()
    plt.show()

print('✔  afficher_v2() définie')

✔  afficher_v2() définie


### `process_and_visualize_v2()` — pipeline complet avec OCR EasyOCR

Mêmes étapes 1-3 que l'original + **étape 4 : OCR v7** sur Diagram et Board,  
puis `afficher_v2()` alimentée par les résultats OCR EasyOCR.

In [20]:
def process_and_visualize_v2(pdf_path_arg, model_path,
                              output_dir='parts/Lecture_des_composants/notebook/results'):
    """
    Version étendue de process_and_visualize() :
    - Étapes 1-3 identiques (chargement, conversion PDF, merge, rotations)
    - Étape 4 (nouvelle) : OCR v7 sur Diagram et Board via run_ocr_on_image()
    - Étape 5 : afficher_v2() alimentée par les résultats OCR v7
    """
    os.makedirs(output_dir, exist_ok=True)

    # ── Étapes 1-3 (identiques à process_and_visualize) ──────────────────────
    try:
        model = YOLO(model_path)
    except Exception as e:
        print(f'Erreur chargement modèle : {e}'); return

    print('--- Conversion du PDF en images ---')
    pages = convert_from_path(pdf_path_arg, dpi=300)
    best_images = {'Table': (None, 0), 'Diagram': (None, 0), 'Board': (None, 0)}

    print('--- Extraction des zones via YOLO ---')
    for page in pages:
        results    = model.predict(source=page, conf=0.212, verbose=False)
        page_array = np.array(page)
        for box in results[0].boxes:
            cls_name = results[0].names[int(box.cls[0])]
            key = {'table': 'Table', 'diagram': 'Diagram', 'board': 'Board'}.get(cls_name.lower())
            if key:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                margin = 10
                crop = page_array[max(0,y1-margin):min(page_array.shape[0],y2+margin),
                                  max(0,x1-margin):min(page_array.shape[1],x2+margin)]
                if crop is not None and crop.size > 0:
                    surf = crop.shape[0] * crop.shape[1]
                    if surf > best_images[key][1]:
                        best_images[key] = (crop, surf)

    temp_paths = {}
    for cls in ['Table', 'Diagram', 'Board']:
        img_data, _ = best_images[cls]
        if img_data is not None:
            img_bgr = cv2.cvtColor(img_data, cv2.COLOR_RGB2BGR)
            if cls == 'Diagram': img_bgr = cv2.rotate(img_bgr, cv2.ROTATE_90_COUNTERCLOCKWISE)
            elif cls == 'Board': img_bgr = cv2.rotate(img_bgr, cv2.ROTATE_90_CLOCKWISE)
            path = f'{output_dir}/temp_{cls.lower()}.png'
            cv2.imwrite(path, img_bgr)
            temp_paths[cls] = path

    if 'Table' not in temp_paths or 'Diagram' not in temp_paths:
        print('\nERREUR : Table ou Diagram manquant.')
        return

    # ── Étape 4 : OCR v7 sur Diagram et Board ────────────────────────────────
    print('\n--- OCR v7 sur Diagram ---')
    ocr_schema = run_ocr_on_image(
        temp_paths['Diagram'], yolo_model, output_dir=output_dir, label='diagram'
    )

    ocr_carte = None
    if 'Board' in temp_paths:
        print('\n--- OCR v7 sur Board ---')
        ocr_carte = run_ocr_on_image(
            temp_paths['Board'], yolo_model, output_dir=output_dir, label='board'
        )

    # ── Étape 5 : Interface interactive ──────────────────────────────────────
    print('\n--- Affichage ---')
    afficher_v2(
        temp_paths['Table'],
        temp_paths['Diagram'],
        temp_paths.get('Board'),
        ocr_schema=ocr_schema,
        ocr_carte=ocr_carte,
    )

print('✔  process_and_visualize_v2() définie')

✔  process_and_visualize_v2() définie


---
### Étape finale : Lancement de l'interface

`Table` et `Diagram` sont obligatoires pour que l'interface ait du sens. `Board` est optionnel.

Deux versions disponibles :
- `process_and_visualize()` — pipeline original (YOLO + Tesseract PSM 7)
- `process_and_visualize_v2()` — pipeline étendu avec **OCR EasyOCR v7** (plus précis)

In [23]:
# Modifiez les chemins selon votre configuration
# Toujours relatifs à la racine du projet (etudes-pratiques/)

pdf_path_str = next(iter(pdf_path)) if pdf_path else 'notebook/uploaded_images/Accuphase-E202amp.pdf'
PDF_PATH   = pdf_path_str
MODEL_PATH = 'models/extraction/best.pt'

# Version originale (YOLO + Tesseract PSM 7) :
# process_and_visualize(PDF_PATH, MODEL_PATH)

# Version étendue avec EasyOCR v7 (recommandée) :
process_and_visualize_v2(PDF_PATH, MODEL_PATH)

--- Conversion du PDF en images ---
--- Extraction des zones via YOLO ---

--- OCR v7 sur Diagram ---
  AUTO  3983×3294px | contraste=79 | luminosité=222 | imgsz=1280 | scale=5 | conf=0.4
  YOLO sur diagram…  300 zones → 274 après merge
  OCR 274/274
  ✔  62 refs valides
  Fichiers : parts/Lecture_des_composants/notebook/results/diagram_20260509_160302_annotated.png

--- OCR v7 sur Board ---
  AUTO  2076×1023px | contraste=67 | luminosité=233 | imgsz=1024 | scale=6 | conf=0.4
  YOLO sur board…  58 zones → 37 après merge
  OCR 37/37
  ✔  3 refs valides
  Fichiers : parts/Lecture_des_composants/notebook/results/board_20260509_160307_annotated.png

--- Affichage ---
Analyse en cours...
 Bilan : Table(11) | Schéma(62) | Carte(3)
Interface prête (OCR v7).
